# Decoder FitLN task-vector analysis

Task vectors (fine-tuned minus pretrained normalization weights) of the decoder FitLN checkpoints. They contain only a scale (gamma) per layer, since RMSNorm has no bias. Checkpoints are read from `out/decoder/fitln/{model}/{task}/{lr}/{seed}/` with `decoder/task_vector_lib.py`, at the learning rate chosen as in `aggregate_decoder_results.py`.

The notebook loads the pretrained 1-4B backbones and the trained `fitln_decoder_weights.bin` files, so it needs a GPU machine. Run it from the repository root or from `notebooks/`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # repo root, so out/decoder/... and analysis/... resolve
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'decoder'))


In [ ]:
import copy


In [ ]:
from pathlib import Path
FIG_DIR = Path('figures/decoder_task_vector')
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path('results/decoder/task_vector')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import math
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

# Deliberately generous font sizes: figure text must stay legible at column width.
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 17,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.titlesize": 18,
})

In [ ]:
import tqdm

In [ ]:
import task_vector_lib as tvl
import common
import run_common


In [ ]:
MODELS = {short: hf for hf, short in common.MODEL_SHORT.items()}
TASKS = common.GLUE_GEN_TASKS + [t for t in common.FREE_GEN_TASKS if t != 'coqa']
EPS = 0.01  # active-channel threshold, matches analysis.task_vectors.sparsity_profile's default

# Reported seeds. Other seed directories (e.g. 0) must be filtered out at discovery,
# because select_best_lr averages the validation loss over the seeds it is given.
VALID_SEEDS = (1, 2, 42)
SEED_SCAN_RANGE = VALID_SEEDS

In [ ]:
unordered_models = copy.copy(MODELS)

In [ ]:
unordered_models

In [ ]:
MODELS = {
    
 'llama3_2_1b': 'meta-llama/Llama-3.2-1B',
 'llama3_2_3b': 'meta-llama/Llama-3.2-3B',
 'qwen2_5_1_5b': 'Qwen/Qwen2.5-1.5B',    
 'qwen2_5_3b': 'Qwen/Qwen2.5-3B',
 'qwen3_5_2b_base': 'Qwen/Qwen3.5-2B-Base',
 'qwen3_5_4b_base': 'Qwen/Qwen3.5-4B-Base'}

In [ ]:
print('Models:', list(MODELS.keys()))
print('Tasks:', TASKS)

## 1. Checkpoints and best learning rate

The learning rate is chosen once per (model, task) by the lowest mean validation `eval_loss` across seeds, as in `aggregate_decoder_results.py`. Seed runs with a non-finite validation loss or checkpoint tensor are dropped (`task_vector_lib.is_valid_seed_run`).

In [ ]:
ALL_REFS = [r for r in tvl.list_fitln_checkpoints(root='out/decoder/fitln')
            if r.seed in VALID_SEEDS and r.task in TASKS]
print(f'{len(ALL_REFS)} FitLN checkpoint refs at seeds {VALID_SEEDS} on tasks {TASKS}')

# the seed filter above must come before select_best_lr
BEST = tvl.select_best_lr(ALL_REFS, require_valid=True)

summary_rows = []
for (model_short, task), refs in sorted(BEST.items()):
    summary_rows.append({
        'model': model_short, 'task': task, 'best_lr': refs[0].lr,
        'n_valid_seeds': len(refs), 'seeds': sorted(r.seed for r in refs),
    })
df_summary = pd.DataFrame(summary_rows).sort_values(['model', 'task']).reset_index(drop=True)
df_summary.to_csv(RESULTS_DIR / 'best_lr_selection.csv', index=False)

# every (model, task) group must have all three seeds
_short = {k: sorted(r.seed for r in refs) for k, refs in BEST.items() if len(refs) != len(VALID_SEEDS)}
assert not _short, f'groups without exactly {len(VALID_SEEDS)} valid seeds: {_short}'

df_summary

Every (model, task) row should show at least 2 valid seeds. If fewer seeds appear than in `results/decoder/global_summary.md`, a seed was dropped by the validity check; inspect its `val_results.json` and `fitln_decoder_weights.bin`.

## 2. Pretrained norm weights

Only `post_attention_layernorm.weight` (per layer) and `model.norm.weight` are kept from each backbone, and the model is deleted right after.

In [ ]:
PRETRAINED_NORMS = {}


def get_pretrained_norms(model_short):
    if model_short in PRETRAINED_NORMS:
        return PRETRAINED_NORMS[model_short]
    hf_id = MODELS[model_short]
    print(f'Loading pretrained backbone: {hf_id} ...')
    model = run_common.load_causal_lm(hf_id)
    norms = tvl.extract_pretrained_norms(model)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    PRETRAINED_NORMS[model_short] = norms
    print(f'  kept {len(norms)} norm tensors, dropped the rest of the backbone')
    return norms

## 3. Task vectors for every (model, task, seed) at the selected learning rate

In [ ]:
TV_CACHE = {}  # (model_short, task, seed) -> TaskVector

for (model_short, task), refs in tqdm.tqdm(BEST.items()):
    pretrained_sd = get_pretrained_norms(model_short)
    for ref in refs:
        finetuned_sd = ref.load()
        tv = tvl.compute_task_vector_decoder(finetuned_sd, pretrained_sd)
        TV_CACHE[(model_short, task, ref.seed)] = tv

print(f'{len(TV_CACHE)} task vectors built')

## A.1 Per-layer magnitude of delta gamma (mean and std across seeds)

In [ ]:
from analysis.task_vectors import magnitude_profile

for model_short in tqdm.tqdm(MODELS):
    rows = []
    for task in TASKS:
        for seed in SEED_SCAN_RANGE:
            tv = TV_CACHE.get((model_short, task, seed))
            if tv is None:
                continue
            prof = magnitude_profile(tv)
            for layer, g in zip(prof['layers'], prof['gamma_norm']):
                rows.append({'task': task, 'layer': layer, 'gamma_norm': g, 'seed': seed})
    if not rows:
        print(f'No data for {model_short}'); continue
    df = pd.DataFrame(rows)
    plt.figure(figsize=(7, 4))
    sns.lineplot(data=df, x='layer', y='gamma_norm', hue='task', errorbar='sd')
    plt.ylabel(r'$\|\Delta\gamma\|_2$ (mean ± std over seeds)'); plt.xlabel('Decoder layer')
    plt.title(f'{model_short}: γ deltas by layer')
    plt.legend(fontsize='small', ncol=2)
    out = FIG_DIR / f'fig_layerwise_magnitude_{model_short}.pdf'
    plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
    print('saved', out)

## A.3 Fraction of channels moved beyond a threshold (mean and std across seeds)

Channels with |delta gamma_i| > `EPS` * max|gamma_pretrained| (the default of `analysis.task_vectors.sparsity_profile`).

In [ ]:
from analysis.task_vectors import sparsity_profile

for model_short in MODELS:
    pretrained_sd = get_pretrained_norms(model_short)
    rows = []
    for task in TASKS:
        for seed in SEED_SCAN_RANGE:
            tv = TV_CACHE.get((model_short, task, seed))
            if tv is None:
                continue
            sp = sparsity_profile(tv, pretrained_sd, eps=EPS)
            for layer, gf in zip(sp['layers'], sp['gamma_active_frac']):
                rows.append({'task': task, 'layer': layer, 'gamma_active': gf, 'seed': seed})
    if not rows:
        print(f'No data for {model_short}'); continue
    df = pd.DataFrame(rows)
    plt.figure(figsize=(7, 3.5))
    sns.lineplot(data=df, x='layer', y='gamma_active', hue='task', errorbar='sd')
    plt.ylabel('γ active fraction (mean ± std over seeds)'); plt.xlabel('Decoder layer')
    plt.title(f'{model_short}: fraction of channels moved beyond {EPS} threshold')
    out = FIG_DIR / f'fig_sparsity_{model_short}.pdf'
    plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
    print('saved', out)

## C.1 Task-vector cosine matrix per model

Off-diagonal: mean cosine similarity between the vectors of two different tasks, over the seeds they share. Diagonal: mean cosine similarity between different seeds of the same task.

In [ ]:
def seed_pairs(seeds):
    seeds = sorted(seeds)
    return [(a, b) for i, a in enumerate(seeds) for b in seeds[i + 1:]]


def cosine_matrix_with_stable_diagonal(model_short, tasks):
    present = [t for t in tasks if any((model_short, t, s) in TV_CACHE for s in SEED_SCAN_RANGE)]
    n = len(present)
    mean_mat = np.full((n, n), np.nan)
    std_mat = np.full((n, n), np.nan)

    seeds_by_task = {
        t: sorted(s for s in SEED_SCAN_RANGE if (model_short, t, s) in TV_CACHE)
        for t in present
    }

    for i, ta in enumerate(present):
        for j, tb in enumerate(present):
            if i == j:
                pairs = seed_pairs(seeds_by_task[ta])
                vals = [tvl.seed_pair_cosine(TV_CACHE[(model_short, ta, a)], TV_CACHE[(model_short, ta, b)])
                        for a, b in pairs]
            else:
                shared = sorted(set(seeds_by_task[ta]) & set(seeds_by_task[tb]))
                vals = [tvl.seed_pair_cosine(TV_CACHE[(model_short, ta, s)], TV_CACHE[(model_short, tb, s)])
                        for s in shared]
            if vals:
                mean_mat[i, j] = float(np.mean(vals))
                std_mat[i, j] = float(np.std(vals))
    return mean_mat, std_mat, present


for model_short in MODELS:
    mean_mat, std_mat, present = cosine_matrix_with_stable_diagonal(model_short, TASKS)
    if len(present) == 0:
        continue
    annot = np.array([[f'{mean_mat[i,j]:.2f}\n±{std_mat[i,j]:.2f}' if not np.isnan(mean_mat[i, j]) else ''
                        for j in range(len(present))] for i in range(len(present))])
    plt.figure(figsize=(1.3 * len(present) + 2, 1.1 * len(present) + 2))
    sns.heatmap(mean_mat, annot=annot, fmt='', xticklabels=present, yticklabels=present,
                cmap='RdBu_r', center=0, vmin=-1, vmax=1)
    plt.title(f'{model_short}: γ task-vector cosine\n(diagonal = same task, cross-seed)')
    out = FIG_DIR / f'fig_cosine_matrix_{model_short}.pdf'
    plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
    print('saved', out)

## E.1 Cross-model magnitude overlay (relative layer depth, mean over seeds)

In [ ]:
for task in tqdm.tqdm(TASKS):
    plt.figure(figsize=(7, 3.5))
    any_data = False
    for model_short in MODELS:
        seeds = sorted(s for s in SEED_SCAN_RANGE if (model_short, task, s) in TV_CACHE)
        if not seeds:
            continue
        any_data = True
        profs = [magnitude_profile(TV_CACHE[(model_short, task, s)]) for s in seeds]
        layers = profs[0]['layers']
        L = max(layers) + 1
        mean_norm = np.mean([p['gamma_norm'] for p in profs], axis=0)
        x = [l / max(1, L - 1) for l in layers]
        plt.plot(x, mean_norm, label=model_short, marker='o', markersize=2)
    if not any_data:
        plt.close(); continue
    plt.xlabel('Relative layer depth'); plt.ylabel(r'$\|\Delta\gamma\|_2$ (mean over seeds)')
    plt.title(f'Cross-model ‖Δγ‖ on {task}')
    plt.legend(fontsize='small')
    out = FIG_DIR / f'fig_crossmodel_gamma_{task}.pdf'
    plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
    print('saved', out)

## F. Seed stability of the salient channels

F.1: cosine similarity of whole vectors, same task and different seeds against different tasks (uses the matrix from C.1).
F.2: per-layer Jaccard similarity of the active-channel sets across seed pairs.

In [ ]:
same_task_rows, diff_task_rows = [], []

for model_short in tqdm.tqdm(MODELS):
    mean_mat, std_mat, present = cosine_matrix_with_stable_diagonal(model_short, TASKS)
    for i, ta in enumerate(present):
        for j, tb in enumerate(present):
            if np.isnan(mean_mat[i, j]):
                continue
            if i == j:
                same_task_rows.append({'model': model_short, 'task': ta, 'cosine': mean_mat[i, j]})
            elif i < j:
                diff_task_rows.append({'model': model_short, 'pair': f'{ta}-{tb}', 'cosine': mean_mat[i, j]})

df_same = pd.DataFrame(same_task_rows).assign(group='same task, different seed')
df_diff = pd.DataFrame(diff_task_rows).assign(group='different task')
df_stability = pd.concat([df_same[['model', 'group', 'cosine']], df_diff[['model', 'group', 'cosine']]],
                          ignore_index=True)

plt.figure(figsize=(9, 4))
sns.stripplot(data=df_stability, x='model', y='cosine', hue='group', dodge=True, alpha=0.7)
sns.pointplot(data=df_stability, x='model', y='cosine', hue='group', dodge=0.4, linestyle='none',
              markers='D', errorbar='sd')
plt.axhline(0.0, color='gray', linestyle='--', lw=0.8)
plt.ylabel('γ task-vector cosine similarity')
plt.title('Seed stability (same task, cross-seed) vs. task discriminability (cross-task)')
plt.xticks(rotation=20)
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title='')
out = FIG_DIR / 'fig_seed_stability_vs_task_discriminability.pdf'
plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
print('saved', out)

In [ ]:
jaccard_rows = []
for model_short in tqdm.tqdm(MODELS):
    pretrained_sd = get_pretrained_norms(model_short)
    for task in TASKS:
        seeds = sorted(s for s in SEED_SCAN_RANGE if (model_short, task, s) in TV_CACHE)
        for a, b in seed_pairs(seeds):
            per_layer = tvl.per_layer_seed_jaccard(
                TV_CACHE[(model_short, task, a)], TV_CACHE[(model_short, task, b)],
                pretrained_sd, eps=EPS,
            )
            for layer, j in per_layer.items():
                jaccard_rows.append({'model': model_short, 'task': task, 'layer': layer, 'jaccard': j})

df_jaccard = pd.DataFrame(jaccard_rows)
df_jaccard.to_csv(RESULTS_DIR / 'per_layer_seed_jaccard.csv', index=False)

for model_short in MODELS:
    sub = df_jaccard[df_jaccard['model'] == model_short]
    if sub.empty:
        continue
    plt.figure(figsize=(7, 3.5))
    sns.lineplot(data=sub, x='layer', y='jaccard', hue='task', errorbar='sd')
    plt.ylabel('Active-channel Jaccard\n(same task, seed pair)'); plt.xlabel('Decoder layer')
    plt.ylim(-0.05, 1.05)
    plt.title(f'{model_short}: are the same γ channels salient across seeds?')
    out = FIG_DIR / f'fig_seed_jaccard_{model_short}.pdf'
    plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.show()
    print('saved', out)

### F.3 Summary table

One row per (model, task): best learning rate, valid seeds, same-task cross-seed cosine, mean cross-task cosine, their gap, and the mean active-channel Jaccard. Saved to `results/decoder/task_vector/seed_stability_summary.csv`.

In [ ]:
rows = []
for model_short in MODELS:
    mean_mat, std_mat, present = cosine_matrix_with_stable_diagonal(model_short, TASKS)
    diff_vals = [mean_mat[i, j] for i in range(len(present)) for j in range(len(present)) if i != j
                 and not np.isnan(mean_mat[i, j])]
    cross_task_baseline = float(np.mean(diff_vals)) if diff_vals else math.nan
    for i, task in tqdm.tqdm(enumerate(present)):
        row = df_summary[(df_summary['model'] == model_short) & (df_summary['task'] == task)]
        best_lr = row['best_lr'].iloc[0] if not row.empty else math.nan
        n_seeds = row['n_valid_seeds'].iloc[0] if not row.empty else 0
        jac_sub = df_jaccard[(df_jaccard['model'] == model_short) & (df_jaccard['task'] == task)]
        rows.append({
            'model': model_short, 'task': task, 'best_lr': best_lr, 'n_valid_seeds': n_seeds,
            'same_task_cosine_mean': mean_mat[i, i], 'same_task_cosine_std': std_mat[i, i],
            'cross_task_cosine_baseline': cross_task_baseline,
            'stability_gap': mean_mat[i, i] - cross_task_baseline,
            'mean_active_channel_jaccard': jac_sub['jaccard'].mean() if not jac_sub.empty else math.nan,
        })

df_stability_summary = pd.DataFrame(rows).sort_values(['model', 'task']).reset_index(drop=True)
df_stability_summary.to_csv(RESULTS_DIR / 'seed_stability_summary.csv', index=False)
df_stability_summary

## Notes

- `post_attention_layernorm` is the norm before the MLP in these models, so it is not equivalent to the encoder's attention-output LayerNorm (see `decoder/norm_freeze.py`).
- With 3 seeds the stability numbers are descriptive only.
- Every number uses the single learning rate chosen per (model, task); there is no learning-rate sweep here.
- Raw magnitudes are not comparable between Qwen3.5 (hybrid attention, larger pretrained RMSNorm scale) and the other models. Compare magnitudes within a family; use the active-channel fraction across architectures.